In [38]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
import pickle
df = pd.read_csv("study_distraction.csv")
df.dropna(inplace=True)
df["Noise_Level"] = df["Noise_Level"].map({"Low": 0, "Medium": 1, "High": 2})
X = df.drop("Distraction_Level", axis=1)
one = OneHotEncoder(sparse_output=False, dtype=int)
oneht = pd.DataFrame(one.fit_transform(X[["Study_Environment"]]), columns=one.get_feature_names_out(["Study_Environment"]), index=X.index)
X = pd.concat([X.drop("Study_Environment", axis=1), oneht], axis=1)
label = LabelEncoder()
Y = label.fit_transform(df["Distraction_Level"])
xtrain, xtest, ytrain, ytest = train_test_split(X, Y, test_size=0.2, random_state=2)
model=LogisticRegression(max_iter=500)
model.fit(xtrain,ytrain)
with open("LogisticRegression.pkl","wb") as file:
    pickle.dump((model,one,label,X.columns.tolist()),file)
ypred = model.predict(xtest)
print("Accuracy:", accuracy_score(ytest, ypred))
print(confusion_matrix(ytest, ypred))
print(classification_report(ytest, ypred, target_names=label.classes_))
sleep=float(input("Enter Sleep Hours: "))
phone=float(input("Enter Phone usage: "))
noise=int(input("Noise Level: "))
study=input("Study Environment: ")
planned=float(input(("Planned Study Hours: ")))
row = pd.DataFrame([[sleep, planned, phone, noise]], columns=["Sleep_Hours", "Planned_Study_Hours", "Phone_Usage_Hours", "Noise_Level"])
env = pd.DataFrame(one.transform(pd.DataFrame({"Study_Environment": [study]})), columns=one.get_feature_names_out(["Study_Environment"]))
new = pd.concat([row, env], axis=1)[X.columns]
predict = label.inverse_transform(model.predict(new))
print("Distraction Level: ", predict[0])

Accuracy: 0.6
[[4 1 0]
 [1 0 2]
 [0 0 2]]
              precision    recall  f1-score   support

        Less       0.80      0.80      0.80         5
      Medium       0.00      0.00      0.00         3
        More       0.50      1.00      0.67         2

    accuracy                           0.60        10
   macro avg       0.43      0.60      0.49        10
weighted avg       0.50      0.60      0.53        10



Enter Sleep Hours:  5
Enter Phone usage:  3
Noise Level:  2
Study Environment:  Home
Planned Study Hours:  3


Distraction Level:  Medium


In [42]:

import pandas as pd
import pickle
with open("LogisticRegression.pkl","rb") as file:
    model,one,label,columns = pickle.load(file)
sleep=float(input("Enter Sleep Hours: "))
phone=float(input("Enter Phone usage: "))
noise=int(input("Noise Level (Low=0, Medium=1, High=2): "))
study=input("Study Environment: ")
planned=float(input("Planned Study Hours: "))
row = pd.DataFrame([[sleep,planned,phone,noise]],columns=["Sleep_Hours","Planned_Study_Hours","Phone_Usage_Hours","Noise_Level"])
env = pd.DataFrame(one.transform(pd.DataFrame({"Study_Environment":[study]})),columns=one.get_feature_names_out(["Study_Environment"]))
new = pd.concat([row,env],axis=1).reindex(columns=columns,fill_value=0)
predict = label.inverse_transform(model.predict(new))
print("Distraction Level:",predict[0])
print("Model Loaded successfully")

Enter Sleep Hours:  4
Enter Phone usage:  5
Noise Level (Low=0, Medium=1, High=2):  2
Study Environment:  Hostel
Planned Study Hours:  7


Distraction Level: More
Model Loaded successfully
